# Hands-On: Choosing the Right Transformer for the Job


Choosing the right transformer model involves matching its capabilities to the task while balancing output quality, efficiency, and computational cost.

In this exercise, we will explore four tasks and learn how to make this decision. For each task, we follow three steps:

1. **Define the task:** Identify the input and expected output.
2. **Choose the model:** Justify the choice based on architecture and cost.
3. **Solve it:** Load the model and run it.

### The decision framework

| Architecture | Attention pattern | Typical output | Typical tasks | Examples |
|---|---|---|---|---|
| Encoder-only | Bidirectional: each token attends to the full input | Contextual representations (token or sentence vectors) | Classification, embeddings, semantic search, NER | BERT, DistilBERT, RoBERTa |
| Decoder-only | Causal: each token attends to itself and preceding tokens | Generated text, one token at a time | Text generation, chat, code, instruction-following | GPT-2, Llama, Qwen |
| Encoder-decoder | Bidirectional encoder; causal decoder with cross-attention to encoder output | Output sequence conditioned on the input | Summarization, translation, text transformation | T5, BART, Marian |

### Cost factors

When choosing a model, we will consider:

- **Parameter count**: affects memory requirement and latency.
- **Inference efficiency:** an encoder answers in one forward pass, while generation runs the decoder once per output token.
- **Reusability**: can we compute something once and reuse it (for example document embeddings)?
- **Specialization**: a small model already fine-tuned for the task often beats a large general model on cost, and sometimes on quality too.

# Setup


### Install and Import Necessary Libraries

In [ ]:
!pip install -q transformers==5.22.0 torch==2.11.0+cpu

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,                       # Bare encoder, returns hidden states
    AutoModelForSequenceClassification,  # Encoder + classification head
    AutoModelForSeq2SeqLM,           # Encoder-decoder with a language-model head
    AutoModelForCausalLM,            # Decoder-only with a language-model head
    set_seed,
)

### Runtime Setup

Set the random seed

In [ ]:
# Set seed for reproducible generation
set_seed(42)

Running on: cpu


# Task 1: Classify Customer Review Sentiment

### Define the Task

- **Input:** A short customer review.
- **Expected output:** `POSITIVE` or `NEGATIVE`, with a confidence score.
- **Challenge:** Sentiment depends on context and negation. For example, "Not good at all" is negative despite containing the word "good".



### Choose the Model

We will use `distilbert-base-uncased-finetuned-sst-2-english`, an encoder-only model fine-tuned for binary sentiment classification.

**Architecture fit**

An encoder is well suited for classification because it processes the entire input bidirectionally before predicting a label.

- Bidirectional attention captures context, including negation and word relationships.
- A classification head maps the text representation to sentiment labels.
- No text generation is required, making a decoder unnecessary.

**Why not the alternatives?**

- **Decoder-only (e.g., GPT-2):** Can classify through prompting, but causal attention limits context at each token, and generating a label adds unnecessary inference cost.
- **Encoder-decoder:** Its generation capabilities are unnecessary for a fixed-label task.

**Cost factors**

- **Parameter count:** DistilBERT has ~66M parameters, 40% fewer than BERT-base, making it faster and it's designed to retain much of BERT-base’s performance while being smaller.
- **Specialization:** Fine-tuned on SST-2 (Stanford Sentiment Treebank), a dataset of movie reviews labeled positive or negative, so no additional training is needed.
- **Inference cost:** Uses one forward pass per batch, allowing efficient classification of multiple reviews.

### Solve It

#### Load the Model

Load the pretrained tokenizer and sentiment classification model.

In [ ]:
# Load a DistilBERT checkpoint fine-tuned for binary sentiment (SST-2)
clf_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_name).to(device)

# Evaluation mode disables dropout during inference
clf_model.eval()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


#### Define Sample Reviews

Include positive, negative, contrastive, and negation examples.

In [ ]:
# Include easy cases and two tricky ones: a contrast sentence and a negation
reviews = [
    "The battery lasts all day and the screen is gorgeous.",
    "Stopped working after two days. Complete waste of money.",
    "I expected it to be terrible, but it turned out to be great.",
    "Not good at all.",
]

#### Tokenize the Reviews

Tokenize the reviews as a batch and move the inputs to the selected device.

In [ ]:
# Tokenize the whole batch at once.
# padding=True pads shorter reviews to the longest one, the attention mask hides the padding.
inputs = clf_tokenizer(reviews, padding=True, truncation=True, return_tensors="pt").to(device)

#### Predict Sentiment

Run inference and convert the model's logits into probabilities and predicted labels.

In [ ]:
# Single forward pass, no gradients needed for inference
with torch.no_grad():
    logits = clf_model(**inputs).logits          # Shape: (num_reviews, num_labels)

# Softmax converts raw logits to probabilities that sum to 1 per review
probs = torch.softmax(logits, dim=-1)
pred_ids = probs.argmax(dim=-1)

#### Display Results

Display each review's predicted sentiment and confidence score.

In [ ]:
# The label names are stored in the model config (0 -> NEGATIVE, 1 -> POSITIVE)
for review, pred, p in zip(reviews, pred_ids, probs):
    label = clf_model.config.id2label[pred.item()]
    print(f"{label:<9} ({p[pred].item():.1%})  {review}")

POSITIVE  (100.0%)  The battery lasts all day and the screen is gorgeous.
NEGATIVE  (100.0%)  Stopped working after two days. Complete waste of money.
POSITIVE  (100.0%)  I expected it to be terrible, but it turned out to be great.
NEGATIVE  (100.0%)  Not good at all.


#### Observation



- The model correctly classifies all four reviews, including those with contrastive wording and negation.
- Encoder-only models are effective for text classification because they use the full input context to predict a label.

# Task 2: Semantic Search Over a Support FAQ

### Define the Task

We need to retrieve FAQ entries that match a user's question based on meaning rather than exact keywords.

- **Input:** A user query and a collection of FAQ entries.
- **Expected output:** FAQ entries ranked by semantic similarity.
- **Challenge:** Queries may express the same intent using different words. For example, "I forgot my login" and "How do I reset my password?" share almost no words but mean the same thing.

### Choose the Model

We will use `sentence-transformers/all-MiniLM-L6-v2`, an encoder-only model trained for sentence embeddings.

**Architecture fit**

- We need one fixed-size vector per text for similarity comparison. The encoder generates contextual token vectors, which we average (mean pool) into a single sentence embedding.
- Unlike raw BERT, this model is trained specifically for semantic similarity, so texts with similar meanings have embeddings closer together in vector space.

**Why not the alternatives?**

- **Decoder-only and encoder-decoder:** Designed primarily for text generation rather than producing comparable sentence embeddings.
- **Cross-encoder:** Can provide more accurate relevance scores but requires a separate forward pass for every query-FAQ pair. A bi-encoder embeds queries and FAQ entries independently, making retrieval more scalable.

**Cost factors**

- **Parameter count:** Only about 22M parameters and 384-dimensional vectors, so it is tiny and fast.
- **Reusability:** FAQ embeddings can be computed once and stored for future queries.


### Solve It

#### Load the Model

In [ ]:
# Load the embedding model. AutoModel returns hidden states (no classification head).
embed_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name).to(device)
embed_model.eval()

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

#### Define the Embedding Function

Tokenize the texts, extract contextual token vectors, mean-pool them while ignoring padding, and normalize the sentence embeddings.

In [ ]:
def embed(texts):
    """Turn a list of strings into L2-normalised sentence embeddings."""
    inputs = embed_tokenizer(
        texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        # One vector per token: shape (batch, seq_len, 384)
        token_embeddings = embed_model(**inputs).last_hidden_state

    # Mean pooling: average the token vectors, but ignore padding tokens.
    # The attention mask is 1 for real tokens and 0 for padding.
    mask = inputs["attention_mask"].unsqueeze(-1).float()     # (batch, seq_len, 1)
    summed = (token_embeddings * mask).sum(dim=1)             # Sum of real tokens only
    counts = mask.sum(dim=1).clamp(min=1e-9)                  # Number of real tokens
    sentence_embeddings = summed / counts                     # (batch, 384)

    # Normalise to unit length so that a dot product equals cosine similarity
    return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)

Define the FAQ entries and compute their embeddings once for reuse.

In [ ]:
# A small FAQ acting as our "document store"
faq = [
    "How do I reset my password?",
    "What is your refund policy for damaged items?",
    "How long does standard shipping take?",
    "Can I change the delivery address after placing an order?",
    "Do you ship to international addresses?",
    "How do I contact customer support by phone?",
]

faq_embeddings = embed(faq)

#### Define User Queries

Use queries phrased differently from the FAQ entries to test semantic matching.

In [ ]:
queries = [
    "I forgot my login details",
    "My package arrived broken, can I get my money back?",
    "Will my order reach me in another country?",
]

query_embeddings = embed(queries)

#### Retrieve Similar FAQs

Compute cosine similarity between each query and FAQ entry, then retrieve the top two matches.

In [ ]:
similarity = query_embeddings @ faq_embeddings.T

for q, scores in zip(queries, similarity):
    top = torch.topk(scores, k=2)
    print(f"Query: {q}")

    for score, idx in zip(top.values, top.indices):
        print(f"   {score.item():.3f}  {faq[idx]}")

    print()

Query: I forgot my login details
   0.584  How do I reset my password?
   0.295  How do I contact customer support by phone?

Query: My package arrived broken, can I get my money back?
   0.533  What is your refund policy for damaged items?
   0.440  How do I contact customer support by phone?

Query: Will my order reach me in another country?
   0.551  Do you ship to international addresses?
   0.446  Can I change the delivery address after placing an order?



#### Observation



- The top-ranked FAQ matches the intent of each query, even when the wording differs.
- Sentence embeddings enable semantic search by comparing meaning rather than exact keyword overlap.

# Task 3: Summarize a News Article

### Define the Task

- **Input**: a multi-sentence article.
- **Expected Output**: a short summary (two to three sentences) that keeps the key facts.
- **Challenge**: the output is a **new piece of text**, not a label, and it must be faithful to the whole input.



### Choose the Model

We will use `sshleifer/distilbart-cnn-12-6`, an encoder-decoder model fine-tuned for summarization.

**Architecture fit**

- Summarization is a sequence-to-sequence task: the model reads the article and generates a summary.
- The encoder processes the article bidirectionally and builds contextual representations. The decoder generates the summary token by token, using cross-attention to access the encoder's output.
- This separates source understanding from text generation, making the architecture well suited for summarization.


**Why not the alternatives?**

- Encoder-only: cannot generate text.
- Decoder-only: can summarize if prompted, and large instruction-tuned models do it well. But the article and summary share one causal stream, and a small general model is less reliable at staying faithful. For a fixed task like this, a specialized model is cheaper and more predictable.

**Cost factors**

- **Parameter count:** DistilBART has a few hundred million parameters, making it larger and slower than the models used in Tasks 1 and 2.
- **Inference efficiency:** It uses 12 encoder layers and 6 decoder layers (hence "12-6"). The decoder runs for each generated token, so reducing decoder layers helps speed up generation.
- **Specialization:** Fine-tuned on CNN/DailyMail for summarization, so no additional training is required.

### Solve It

#### Load the Model

Load the pretrained tokenizer and sequence-to-sequence model.

In [ ]:
# Load a DistilBART checkpoint already fine-tuned for news summarization
sum_name = "sshleifer/distilbart-cnn-12-6"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name).to(device)
sum_model.eval()

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

#### Define a News Article

Provide a multi-sentence article for summarization.

In [ ]:
article = """
The city council of Riverton approved a plan on Tuesday to expand its bike-share program
from 40 stations to 150 stations over the next three years. The project will cost 12 million
dollars, with half of the funding coming from a regional transport grant. Officials said the
expansion will focus on neighborhoods that currently have limited bus service, and that
protected bike lanes will be built along the busiest routes. Supporters argued that the plan
will reduce car traffic and improve air quality, while some business owners raised concerns
about the loss of on-street parking. The council agreed to review the impact on local
businesses after the first year and adjust the plan if needed. Construction is expected to
begin in March.
"""

#### Generate the Summary

Tokenize the article and generate a summary of two to three sentences.

In [ ]:
# Encode the article. truncation=True cuts anything beyond the model's 1024 token limit for this checkpoint.
inputs = sum_tokenizer(article, return_tensors="pt", max_length=1024, truncation=True).to(device)

# Generate the summary
summary_ids = sum_model.generate(
    **inputs,
    num_beams=4,          # Beam search keeps 4 candidate summaries, giving more coherent text
    max_length=60,        # Upper bound on summary length (in tokens)
    min_length=20,        # Prevents trivially short summaries
    early_stopping=True,  # Stop once all beams have finished
)

summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

#### Display the Result

Print the generated summary.

In [ ]:
print("Summary:", summary)
print()
print(f"Article: {len(article.split())} words -> Summary: {len(summary.split())} words")

Summary:  The city council of Riverton approved a plan on Tuesday to expand its bike-share program from 40 stations to 150 stations over the next three years . The project will cost 12 million dollars, with half of the funding coming from a regional transport grant .

Article: 122 words -> Summary: 46 words


### Observation



- The summary captures the expansion plan, timeline, and funding details, while omitting neighborhood priorities and business concerns.
- Encoder-decoder models are well suited for summarization because they encode the source text and generate a shorter version.

# Task 4: Draft a Customer Support Reply

### Define the Task

- **Input:** An instruction describing the situation, tone, length, and required details.
- **Expected output:** A customer-facing reply that follows all instructions.
- **Challenge:** The response is open-ended, with no fixed labels or source text to transform.

### Choose the Model

We will use `Qwen/Qwen2.5-0.5B-Instruct`, a small instruction-tuned decoder-only model.

**Architecture fit**

- Decoder-only models predict the next token based on the preceding text, making them well suited for text generation.
- Instruction tuning helps the model follow requests such as “under 80 words, apologize, and ask for the order number.”
- The same model can perform different tasks through prompting, making it suitable for general-purpose assistants.

**Why not the alternatives?**
- **Encoder-only:** cannot generate text.
- **Encoder-decoder:** Can generate text, but a separate encoder is less necessary when the input is a short instruction rather than a document to transform.

**Cost factors**

- **Inference efficiency:** Generation is autoregressive, requiring one forward pass per output token. `max_new_tokens` limits reply length and cost.
- **Parameter count:** At around 0.5B parameters. Larger models require substantially more memory.
- **Trade-off:** Smaller models are cheaper but less capable, so customer-facing replies should be reviewed before sending.

### Solve It

#### Load the Model

Load the pretrained tokenizer and instruction-tuned causal language model.

In [ ]:
# Load a small instruction-tuned decoder-only model
gen_name = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_name).to(device)
gen_model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

#### Define the Instruction

Provide the customer scenario and specify the tone, length, and required details.

In [ ]:
# Instruction-tuned models expect a chat structure: a system role and a user role
messages = [
    {"role": "system", "content": "You are a courteous customer support agent for an online electronics store."},
    {"role": "user", "content": (
        "Write a short, polite reply (under 80 words) to a customer whose headphones "
        "arrived with a cracked ear cup. Apologize, offer a free replacement, "
        "and ask for their order number."
    )},
]

### Format the prompt

In [ ]:
# The chat template wraps the messages in the special tokens this model was trained with.
# add_generation_prompt=True appends the marker that tells the model to start its reply.
prompt_text = gen_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("Formatted prompt:\n", prompt_text)

Formatted prompt:
 <|im_start|>system
You are a courteous customer support agent for an online electronics store.<|im_end|>
<|im_start|>user
Write a short, polite reply (under 80 words) to a customer whose headphones arrived with a cracked ear cup. Apologize, offer a free replacement, and ask for their order number.<|im_end|>
<|im_start|>assistant



#### Generate the Reply

Generate a response with a token limit and sampling settings.

In [ ]:
inputs = gen_tokenizer(prompt_text, return_tensors="pt").to(device)

# Sampling with moderate temperature gives natural, slightly varied wording.
# max_new_tokens caps the number of generated tokens, which also caps the cost.
output_ids = gen_model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7,   # Lower is more predictable, higher is more creative
    top_p=0.9,         # Sample only from the smallest set of tokens covering 90% probability
)

# The output contains the prompt followed by the reply, so we slice off the prompt tokens
new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

#### Display the Result

Decode and print only the newly generated reply.

In [ ]:
reply = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Reply:\n", reply)

Reply:
 Dear [Customer],

Thank you for your prompt feedback regarding the damaged headphones. We are deeply sorry that this has occurred. We appreciate your patience and understanding.

We kindly request that you contact our customer service team immediately to arrange for a free replacement of the ear cups within the next 24 hours. Please include your order number so we can ensure it is handled promptly.

Your satisfaction is our top priority at [Electronics Store Name]. Thank you once again for your support.

Best regards,
[Your Name]  
[Contact Information]


### Observation



- The reply is polite and apologetic, but exceeds the 80-word limit and does not clearly offer a free replacement of the headphones.
- Decoder-only models can generate flexible responses, but outputs should be checked against the task requirements.

# Key Insights: Choosing a Model

Ask these questions in order:

1. **Is the output a label, score, or vector?** Use an **encoder-only** model. Use a classification head for labels, or pool hidden states to create embeddings.
2. **Is the output new text derived from a specific input (summary, translation, rewrite)?** Use an **encoder-decoder** model, especially when a task-specific fine-tuned checkpoint is available.
3. **Is the output open-ended text, or must the model follow varied instructions?** Use a **decoder-only** model, choosing the smallest size that meets your quality requirements.

Then check the cost factors:

| Factor | Question to ask |
|---|---|
| Parameter count | Can a smaller model meet the quality bar while fitting the memory budget? |
| Inference efficiency | How much computation is needed to produce the output? |
| Reusability | Can results, such as document embeddings, be precomputed and cached? |
| Task specialization | Is a checkpoint already fine-tuned for this task? |
| Input length | How will longer inputs affect memory and latency? |

**Rule of thumb:** Choose the simplest architecture that fits the output type, then the smallest model that meets the quality bar. A large decoder-only model can perform all four tasks, but may cost much more with little or no benefit on the first three.

<font size=6 color="#4682B4">Power Ahead!</font>
___